# Phase II — CRCS BCI Security Benchmark

**Google Colab Master — PHASE2-COLAB-2.2**

This version fixes the Step-6 `EmptyDataError` seen in quick mode.

### Why that error occurred
Quick mode deliberately runs only Subject 1. Inferential Wilcoxon testing requires at least 5 paired subjects, so no statistical test should be produced at that stage. Version 2.2 now writes a header-only statistics CSV and the inspection cell reports “not available in quick mode” instead of crashing.

### Final calibration grid
**0%, 10%, 25%, 50%, 75%, 80%**

### Full experiment
9 subjects × 5 seeds × 6 calibration levels × 2 alignment modes × 4 scenarios = **2160 primary rows**.


In [ ]:
# STEP 1 — INSTALL COLAB DEPENDENCIES
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "moviepy"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
packages=["jedi>=0.16","moabb==1.5.0","mne==1.12.1","numpy>=1.26","scipy>=1.13","pandas>=2.1","scikit-learn>=1.5","matplotlib>=3.8"]
subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade-strategy","only-if-needed",*packages],check=True)
print("✅ Dependencies installed.")
print("If Colab asks for a restart, restart once and continue from STEP 2.")


✅ Dependencies installed.
If Colab asks for a restart, restart once and continue from STEP 2.


If Colab requests a restart after STEP 1, restart once and continue from STEP 2.


In [ ]:
# STEP 2 — VERIFY ENVIRONMENT AND DATASET
import sys, numpy as np, scipy, pandas as pd, sklearn, matplotlib, mne, moabb
from moabb.datasets import BNCI2014_001
print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("SciPy        :", scipy.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("MNE          :", mne.__version__)
print("MOABB        :", moabb.__version__)
ds=BNCI2014_001(); assert list(ds.subject_list)==list(range(1,10))
print("✅ BNCI2014_001 subjects:",ds.subject_list)


Python       : 3.13.15
NumPy        : 2.1.3
SciPy        : 1.16.3
Pandas       : 2.2.3
Scikit-learn : 1.6.1
Matplotlib   : 3.10.0
MNE          : 1.12.1
MOABB        : 1.5.0
✅ BNCI2014_001 subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9]


In [ ]:
# STEP 3 — CREATE EXACT v2.2 SCRIPT
from pathlib import Path
import hashlib

SCRIPT = '#!/usr/bin/env python3\n"""\nCross-session, calibration-efficient, security-robust MI-BCI benchmark.\n\nResearch question\n-----------------\nHow do clean decoding performance and targeted backdoor vulnerability change when\ncross-session target-label calibration is reduced, and can calibration-referenced\nclass-consistency screening suppress dirty-label backdoors without sacrificing\nheld-out target-session utility?\n\nDataset\n-------\nBCI Competition IV Dataset 2a (MOABB: BNCI2014_001): 9 subjects, 2 sessions,\n22 EEG channels, 250 Hz, 4 motor-imagery classes.\n\nIMPORTANT\n---------\nThis script is a Phase-II experimental benchmark scaffold. The proposed\ncalibration-referenced class-consistency screening (CRCS) is a candidate method to\nbe validated, NOT a novelty claim. Every table/plot is computed from executed\nexperiments; there are no hard-coded performance values.\n\nRecommended environment: Python 3.11+, MOABB 1.5.0, MNE 1.12.1.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport warnings\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nPACKAGE_VERSION = "PHASE2-COLAB-2.2"\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import welch\nfrom scipy.integrate import trapezoid\nfrom scipy.stats import wilcoxon\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    cohen_kappa_score,\n    f1_score,\n)\nfrom sklearn.preprocessing import LabelEncoder\n\ntry:\n    import mne\n    from mne.decoding import CSP\n    mne.set_log_level("WARNING")\nexcept Exception as exc:  # pragma: no cover\n    raise RuntimeError(\n        "MNE is required. Install the tested environment with:\\n"\n        "  python -m pip install -r requirements_colab.txt"\n    ) from exc\n\n\n# -----------------------------------------------------------------------------\n# Reproducibility and configuration\n# -----------------------------------------------------------------------------\n\n\ndef set_global_seed(seed: int) -> None:\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n\n\n@dataclass(frozen=True)\nclass Config:\n    subjects: Tuple[int, ...] = tuple(range(1, 10))\n    seeds: Tuple[int, ...] = (11, 22, 33, 44, 55)\n    calibration_fractions: Tuple[float, ...] = (0.0, 0.10, 0.25, 0.50, 0.75, 0.80)\n    alignment_modes: Tuple[str, ...] = ("inductive", "transductive")\n    fmin: float = 8.0\n    fmax: float = 30.0\n    target_class_name: str = "left_hand"\n    poison_rate: float = 0.10\n    trigger_frequency_hz: float = 23.0\n    trigger_amplitude_std: float = 0.55\n    trigger_channels: Tuple[str, ...] = ("C3", "Cz", "C4")\n    csp_components: int = 8\n\n    # Phase-I spectral sanitizer retained as an explicit baseline.\n    sanitizer_z_threshold: float = 3.5\n    sanitizer_max_remove_fraction: float = 0.15\n    min_class_samples_after_sanitize: int = 8\n\n    # Phase-II candidate defense: calibration-referenced class-consistency screening.\n    crcs_max_remove_fraction: float = 0.15\n    crcs_min_class_samples: int = 8\n    crcs_min_reference_per_class: int = 3\n    crcs_margin_threshold: float = 0.0\n    crcs_spectral_weight: float = 0.25\n    crcs_spectral_z_threshold: float = 2.5\n    crcs_subbands: Tuple[Tuple[float, float], ...] = (\n        (8.0, 12.0),\n        (12.0, 16.0),\n        (16.0, 20.0),\n        (20.0, 24.0),\n        (24.0, 30.0),\n    )\n\n    output_dir: str = "bci_security_phase2_results"\n\n\ndef validate_config(cfg: Config) -> None:\n    if not cfg.subjects:\n        raise ValueError("At least one subject is required.")\n    if not cfg.seeds:\n        raise ValueError("At least one seed is required.")\n    if not all(0.0 <= f < 1.0 for f in cfg.calibration_fractions):\n        raise ValueError("Calibration fractions must satisfy 0 <= f < 1.")\n    if not 0.0 < cfg.poison_rate < 0.5:\n        raise ValueError("poison_rate should be in (0, 0.5) for this benchmark.")\n    if cfg.fmin <= 0 or cfg.fmax <= cfg.fmin:\n        raise ValueError("Invalid frequency band.")\n    if cfg.trigger_frequency_hz <= cfg.fmin or cfg.trigger_frequency_hz >= cfg.fmax:\n        raise ValueError(\n            "Trigger frequency must lie inside the analysis passband so that the "\n            "attack is not trivially removed by preprocessing."\n        )\n    allowed_modes = {"inductive", "transductive"}\n    if not cfg.alignment_modes or any(m not in allowed_modes for m in cfg.alignment_modes):\n        raise ValueError(\n            f"alignment_modes must be drawn from {sorted(allowed_modes)}; "\n            f"got {cfg.alignment_modes}"\n        )\n    if not 0.0 < cfg.crcs_max_remove_fraction < 0.5:\n        raise ValueError("crcs_max_remove_fraction must lie in (0, 0.5).")\n\n\n# -----------------------------------------------------------------------------\n# Linear algebra: Euclidean Alignment (EA)\n# -----------------------------------------------------------------------------\n\n\ndef _trial_covariance(x: np.ndarray, eps: float = 1e-10) -> np.ndarray:\n    """Trace-normalized covariance of one trial, x shape = channels x samples."""\n    if x.ndim != 2:\n        raise ValueError("A trial must have shape (channels, samples).")\n    xc = x - x.mean(axis=1, keepdims=True)\n    c = (xc @ xc.T) / max(1, xc.shape[1] - 1)\n    tr = float(np.trace(c))\n    if tr <= eps:\n        c = c + eps * np.eye(c.shape[0])\n        tr = float(np.trace(c))\n    return c / tr\n\n\ndef fit_euclidean_alignment(X: np.ndarray, eps: float = 1e-7) -> np.ndarray:\n    """Fit W = R_bar^{-1/2}; X shape = trials x channels x samples."""\n    if X.ndim != 3 or len(X) < 2:\n        raise ValueError("X must be 3-D with at least two trials.")\n    covs = np.stack([_trial_covariance(x) for x in X], axis=0)\n    r_bar = covs.mean(axis=0)\n    r_bar = 0.5 * (r_bar + r_bar.T) + eps * np.eye(r_bar.shape[0])\n    vals, vecs = np.linalg.eigh(r_bar)\n    vals = np.clip(vals, eps, None)\n    inv_sqrt = (vecs * (1.0 / np.sqrt(vals))) @ vecs.T\n    return 0.5 * (inv_sqrt + inv_sqrt.T)\n\n\ndef apply_alignment(X: np.ndarray, W: np.ndarray) -> np.ndarray:\n    if X.ndim != 3 or W.ndim != 2:\n        raise ValueError("Expected X=(trials,channels,samples), W=(channels,channels).")\n    return np.einsum("ij,njt->nit", W, X, optimize=True)\n\n\n# -----------------------------------------------------------------------------\n# Targeted spectral backdoor attack\n# -----------------------------------------------------------------------------\n\n\ndef resolve_channel_indices(ch_names: Sequence[str], preferred: Sequence[str]) -> List[int]:\n    mapping = {str(name).strip().lower(): i for i, name in enumerate(ch_names)}\n    out = [mapping[name.lower()] for name in preferred if name.lower() in mapping]\n    if out:\n        return sorted(set(out))\n    # Safe fallback: central 3 channels if names are unavailable/unexpected.\n    n = len(ch_names)\n    mid = n // 2\n    return sorted(set([max(0, mid - 1), mid, min(n - 1, mid + 1)]))\n\n\ndef add_spectral_trigger(\n    X: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> np.ndarray:\n    """Add a deterministic narrow-band trigger scaled to each trial/channel SD."""\n    X2 = np.array(X, dtype=np.float64, copy=True)\n    n_times = X2.shape[-1]\n    t = np.arange(n_times, dtype=np.float64) / float(sfreq)\n    wave = np.sin(2.0 * np.pi * float(frequency_hz) * t)\n\n    for idx in channel_indices:\n        if idx < 0 or idx >= X2.shape[1]:\n            raise IndexError(f"Channel index {idx} out of range for {X2.shape[1]} channels.")\n        scale = np.std(X2[:, idx, :], axis=1, ddof=1)\n        scale = np.maximum(scale, np.finfo(np.float64).eps)\n        X2[:, idx, :] += amplitude_std * scale[:, None] * wave[None, :]\n    return X2\n\n\ndef poison_training_set(\n    X: np.ndarray,\n    y: np.ndarray,\n    target_class: int,\n    poison_rate: float,\n    rng: np.random.Generator,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Dirty-label targeted poisoning: trigger non-target trials and relabel target."""\n    eligible = np.flatnonzero(y != target_class)\n    if len(eligible) == 0:\n        raise ValueError("No non-target trials are available for poisoning.")\n\n    n_poison = max(1, int(round(poison_rate * len(y))))\n    n_poison = min(n_poison, len(eligible))\n    chosen = rng.choice(eligible, size=n_poison, replace=False)\n\n    Xp = np.array(X, copy=True)\n    yp = np.array(y, copy=True)\n    Xp[chosen] = add_spectral_trigger(\n        Xp[chosen], sfreq, channel_indices, frequency_hz, amplitude_std\n    )\n    yp[chosen] = target_class\n\n    mask = np.zeros(len(y), dtype=bool)\n    mask[chosen] = True\n    return Xp, yp, mask\n\n\n# -----------------------------------------------------------------------------\n# Generic spectral-outlier sanitization baseline\n# -----------------------------------------------------------------------------\n\n\ndef spectral_concentration_score(\n    X: np.ndarray,\n    sfreq: float,\n    fmin: float,\n    fmax: float,\n    channel_indices: Optional[Sequence[int]] = None,\n) -> np.ndarray:\n    """\n    Trigger-agnostic trial score.\n\n    It does NOT use the attack frequency. For each trial it measures the strongest\n    narrow spectral concentration anywhere inside the analysis band, then takes\n    the maximum across selected channels.\n    """\n    if X.ndim != 3:\n        raise ValueError("X must be 3-D.")\n    if channel_indices is None:\n        Xi = X\n    else:\n        Xi = X[:, list(channel_indices), :]\n\n    nperseg = min(512, Xi.shape[-1])\n    freqs, psd = welch(\n        Xi,\n        fs=float(sfreq),\n        nperseg=nperseg,\n        noverlap=nperseg // 2,\n        axis=-1,\n        detrend="constant",\n        scaling="density",\n    )\n    band = (freqs >= fmin) & (freqs <= fmax)\n    if band.sum() < 3:\n        raise ValueError("Too few Welch frequency bins in the analysis band.")\n\n    p = np.maximum(psd[..., band], np.finfo(np.float64).tiny)\n    concentration = p.max(axis=-1) / p.sum(axis=-1)\n    return concentration.max(axis=1)\n\n\ndef robust_sanitize(\n    X: np.ndarray,\n    y: np.ndarray,\n    scores: np.ndarray,\n    z_threshold: float,\n    max_remove_fraction: float,\n    min_class_samples: int,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:\n    """Remove high-score spectral outliers using median/MAD with a removal cap."""\n    if not (len(X) == len(y) == len(scores)):\n        raise ValueError("X, y and scores must have equal length.")\n\n    median = float(np.median(scores))\n    mad = float(np.median(np.abs(scores - median)))\n    robust_sigma = 1.4826 * mad\n    if robust_sigma <= np.finfo(np.float64).eps:\n        robust_z = np.zeros_like(scores, dtype=float)\n    else:\n        robust_z = (scores - median) / robust_sigma\n\n    candidates = np.flatnonzero(robust_z > z_threshold)\n    max_remove = int(math.floor(max_remove_fraction * len(y)))\n    if max_remove > 0 and len(candidates) > max_remove:\n        order = candidates[np.argsort(scores[candidates])[::-1]]\n        candidates = order[:max_remove]\n\n    keep = np.ones(len(y), dtype=bool)\n    keep[candidates] = False\n\n    # Guarantee enough examples remain in every class.\n    for cls in np.unique(y):\n        cls_idx = np.flatnonzero(y == cls)\n        kept_count = int(np.sum(keep[cls_idx]))\n        if kept_count < min_class_samples:\n            need = min(min_class_samples - kept_count, int(np.sum(~keep[cls_idx])))\n            if need > 0:\n                removed_cls = cls_idx[~keep[cls_idx]]\n                restore = removed_cls[np.argsort(scores[removed_cls])[:need]]\n                keep[restore] = True\n\n    stats = {\n        "score_median": median,\n        "score_mad": mad,\n        "removed_count": int((~keep).sum()),\n        "removed_fraction": float((~keep).mean()),\n    }\n    return X[keep], y[keep], keep, stats\n\n\n\n# -----------------------------------------------------------------------------\n# Phase-II candidate defense:\n# Calibration-Referenced Class-Consistency Screening (CRCS)\n# -----------------------------------------------------------------------------\n\n\ndef _relative_bandpower_features(\n    X: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    subbands: Sequence[Tuple[float, float]],\n) -> np.ndarray:\n    """\n    Generic trigger-agnostic descriptor.\n\n    For each selected motor channel, compute relative Welch power in fixed\n    sub-bands spanning the full 8-30 Hz MI band. The implementation never uses\n    the attack frequency. Shape: trials x (channels * subbands).\n    """\n    if X.ndim != 3:\n        raise ValueError("X must have shape (trials, channels, samples).")\n    if not channel_indices:\n        raise ValueError("At least one channel is required.")\n    Xi = X[:, list(channel_indices), :]\n    nperseg = min(512, Xi.shape[-1])\n    freqs, psd = welch(\n        Xi,\n        fs=float(sfreq),\n        nperseg=nperseg,\n        noverlap=nperseg // 2,\n        axis=-1,\n        detrend="constant",\n        scaling="density",\n    )\n    total_band = (freqs >= min(b[0] for b in subbands)) & (\n        freqs <= max(b[1] for b in subbands)\n    )\n    total = trapezoid(psd[..., total_band], freqs[total_band], axis=-1)\n    total = np.maximum(total, np.finfo(np.float64).tiny)\n\n    feats = []\n    for lo, hi in subbands:\n        mask = (freqs >= float(lo)) & (freqs < float(hi))\n        if int(mask.sum()) < 2:\n            raise ValueError(f"Too few frequency bins for CRCS subband {(lo, hi)}.")\n        bp = trapezoid(psd[..., mask], freqs[mask], axis=-1)\n        rel = np.maximum(bp / total, np.finfo(np.float64).tiny)\n        feats.append(np.log(rel))\n    # trials x channels x bands -> trials x flat features\n    F = np.stack(feats, axis=-1).reshape(len(X), -1)\n    return np.asarray(F, dtype=np.float64)\n\n\ndef _robust_location_scale(F: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    med = np.median(F, axis=0)\n    mad = np.median(np.abs(F - med), axis=0)\n    scale = 1.4826 * mad\n    # Avoid zero-MAD dimensions without introducing data-dependent explosions.\n    fallback = np.std(F, axis=0, ddof=1) if len(F) > 1 else np.ones(F.shape[1])\n    scale = np.where(scale > 1e-8, scale, np.maximum(fallback, 1e-6))\n    return med, scale\n\n\ndef _standardize_with_reference(\n    F_reference: np.ndarray, F_other: np.ndarray\n) -> Tuple[np.ndarray, np.ndarray]:\n    med, scale = _robust_location_scale(F_reference)\n    return (F_reference - med) / scale, (F_other - med) / scale\n\n\ndef _class_prototypes(F: np.ndarray, y: np.ndarray) -> Dict[int, np.ndarray]:\n    protos: Dict[int, np.ndarray] = {}\n    for cls in np.unique(y):\n        idx = np.flatnonzero(y == cls)\n        if len(idx):\n            protos[int(cls)] = np.median(F[idx], axis=0)\n    return protos\n\n\ndef _prototype_distances(\n    F: np.ndarray, prototypes: Dict[int, np.ndarray]\n) -> Dict[int, np.ndarray]:\n    out: Dict[int, np.ndarray] = {}\n    for cls, proto in prototypes.items():\n        # Mean absolute robust-standardized distance is stable in small calibration sets.\n        out[int(cls)] = np.mean(np.abs(F - proto[None, :]), axis=1)\n    return out\n\n\ndef _class_mismatch_margin(\n    F: np.ndarray, y_claimed: np.ndarray, prototypes: Dict[int, np.ndarray]\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """\n    Margin > 0 means the trial is closer to another class prototype than to its\n    claimed class. This is useful for dirty-label poisoning without knowing the\n    attack target class.\n    """\n    dists = _prototype_distances(F, prototypes)\n    classes = sorted(prototypes)\n    claimed = np.full(len(F), np.nan, dtype=float)\n    best_other = np.full(len(F), np.inf, dtype=float)\n\n    for i, cls in enumerate(y_claimed):\n        c = int(cls)\n        if c not in dists:\n            continue\n        claimed[i] = dists[c][i]\n        alt = [dists[k][i] for k in classes if k != c]\n        if alt:\n            best_other[i] = float(np.min(alt))\n    margin = claimed - best_other\n    return margin, claimed, best_other\n\n\ndef calibration_referenced_consistency_screen(\n    X_source: np.ndarray,\n    y_source_claimed: np.ndarray,\n    X_reference: np.ndarray,\n    y_reference: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    subbands: Sequence[Tuple[float, float]],\n    max_remove_fraction: float,\n    min_class_samples: int,\n    min_reference_per_class: int,\n    margin_threshold: float,\n    spectral_weight: float,\n    spectral_z_threshold: float,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:\n    """\n    CRCS candidate defense.\n\n    Reference hierarchy:\n    1) labeled clean target-session calibration data when every class has enough\n       examples;\n    2) otherwise the source set itself, using robust medians (zero-calibration\n       fallback).\n\n    The screen is attack-frequency agnostic:\n    - class-consistency term: source trial vs. class prototypes;\n    - spectral term: generic narrow-band concentration relative to the reference\n      distribution for the claimed class.\n\n    Samples are removed only when the combined evidence is suspicious, with a\n    global removal cap and per-class minimum sample guarantee.\n    """\n    if len(X_source) != len(y_source_claimed):\n        raise ValueError("Source X/y length mismatch.")\n    if len(X_reference) != len(y_reference):\n        raise ValueError("Reference X/y length mismatch.")\n\n    classes = np.unique(y_source_claimed)\n    ref_ok = len(X_reference) > 0 and all(\n        np.sum(y_reference == cls) >= min_reference_per_class for cls in classes\n    )\n    if ref_ok:\n        Xref, yref = X_reference, y_reference\n        reference_kind = "target_calibration"\n    else:\n        Xref, yref = X_source, y_source_claimed\n        reference_kind = "source_robust_fallback"\n\n    F_ref = _relative_bandpower_features(\n        Xref, sfreq, channel_indices, subbands\n    )\n    F_src = _relative_bandpower_features(\n        X_source, sfreq, channel_indices, subbands\n    )\n    F_ref_z, F_src_z = _standardize_with_reference(F_ref, F_src)\n    protos = _class_prototypes(F_ref_z, yref)\n\n    # If a class is missing from the reference, fall back to source prototypes.\n    missing = [int(c) for c in classes if int(c) not in protos]\n    if missing:\n        F_all_ref_z, F_all_src_z = _standardize_with_reference(F_src, F_src)\n        source_protos = _class_prototypes(F_all_ref_z, y_source_claimed)\n        for c in missing:\n            protos[c] = source_protos[c]\n        # Keep original source feature coordinates; missing-class fallback is rare.\n        # The cap/min-count safeguards prevent catastrophic removal.\n\n    margin, d_claim, _ = _class_mismatch_margin(\n        F_src_z, y_source_claimed, protos\n    )\n\n    # Generic spectral concentration (does not know trigger frequency).\n    src_conc = spectral_concentration_score(\n        X_source,\n        sfreq=sfreq,\n        fmin=min(b[0] for b in subbands),\n        fmax=max(b[1] for b in subbands),\n        channel_indices=channel_indices,\n    )\n    ref_conc = spectral_concentration_score(\n        Xref,\n        sfreq=sfreq,\n        fmin=min(b[0] for b in subbands),\n        fmax=max(b[1] for b in subbands),\n        channel_indices=channel_indices,\n    )\n\n    spectral_z = np.zeros(len(X_source), dtype=float)\n    for cls in classes:\n        src_i = np.flatnonzero(y_source_claimed == cls)\n        ref_i = np.flatnonzero(yref == cls)\n        if len(ref_i) < 2:\n            ref_i = np.arange(len(yref))\n        vals = ref_conc[ref_i]\n        med = float(np.median(vals))\n        mad = float(np.median(np.abs(vals - med)))\n        sigma = max(1.4826 * mad, float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0, 1e-8)\n        spectral_z[src_i] = (src_conc[src_i] - med) / sigma\n\n    # Positive class mismatch is primary; narrow-band excess is secondary.\n    positive_margin = np.maximum(margin - float(margin_threshold), 0.0)\n    spectral_excess = np.maximum(\n        spectral_z - float(spectral_z_threshold), 0.0\n    )\n    suspicion = positive_margin + float(spectral_weight) * spectral_excess\n\n    candidates = np.flatnonzero(suspicion > 0.0)\n    max_remove = int(math.floor(max_remove_fraction * len(y_source_claimed)))\n    if max_remove > 0 and len(candidates) > max_remove:\n        order = candidates[np.argsort(suspicion[candidates])[::-1]]\n        candidates = order[:max_remove]\n\n    keep = np.ones(len(y_source_claimed), dtype=bool)\n    keep[candidates] = False\n\n    # Preserve enough examples per class.\n    for cls in classes:\n        cls_idx = np.flatnonzero(y_source_claimed == cls)\n        if int(np.sum(keep[cls_idx])) < min_class_samples:\n            need = min_class_samples - int(np.sum(keep[cls_idx]))\n            removed_cls = cls_idx[~keep[cls_idx]]\n            if len(removed_cls):\n                restore = removed_cls[np.argsort(suspicion[removed_cls])[:need]]\n                keep[restore] = True\n\n    stats = {\n        "reference_target_calibration": float(reference_kind == "target_calibration"),\n        "removed_count": int(np.sum(~keep)),\n        "removed_fraction": float(np.mean(~keep)),\n        "mean_suspicion": float(np.nanmean(suspicion)),\n        "max_suspicion": float(np.nanmax(suspicion)),\n        "mean_margin": float(np.nanmean(margin)),\n        "mean_spectral_z": float(np.nanmean(spectral_z)),\n    }\n    return X_source[keep], y_source_claimed[keep], keep, stats\n\n\ndef choose_target_alignment(\n    alignment_mode: str,\n    X_source_reference: np.ndarray,\n    X_target_all: np.ndarray,\n    X_target_calibration: np.ndarray,\n) -> np.ndarray:\n    """\n    transductive: use all unlabeled target trials (Phase-I setting).\n    inductive: use calibration trials only; at 0% calibration use source alignment.\n    """\n    if alignment_mode == "transductive":\n        return fit_euclidean_alignment(X_target_all)\n    if alignment_mode == "inductive":\n        if len(X_target_calibration) >= 2:\n            return fit_euclidean_alignment(X_target_calibration)\n        return fit_euclidean_alignment(X_source_reference)\n    raise ValueError(f"Unknown alignment_mode={alignment_mode!r}")\n\n\n\n# -----------------------------------------------------------------------------\n# Classifier and metrics\n# -----------------------------------------------------------------------------\n\n\nclass CSPLDA:\n    """Small deterministic CSP + shrinkage-LDA baseline for 4-class MI."""\n\n    def __init__(self, n_components: int = 8):\n        self.n_components = int(n_components)\n        self.csp: Optional[CSP] = None\n        self.lda: Optional[LinearDiscriminantAnalysis] = None\n\n    def fit(self, X: np.ndarray, y: np.ndarray) -> "CSPLDA":\n        n_components = max(2, min(self.n_components, X.shape[1]))\n        self.csp = CSP(\n            n_components=n_components,\n            reg="ledoit_wolf",\n            log=True,\n            norm_trace=False,\n            transform_into="average_power",\n            rank=None,\n        )\n        self.lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", category=RuntimeWarning)\n            F = self.csp.fit_transform(X, y)\n        self.lda.fit(F, y)\n        return self\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        if self.csp is None or self.lda is None:\n            raise RuntimeError("Model has not been fitted.")\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", category=RuntimeWarning)\n            F = self.csp.transform(X)\n        return self.lda.predict(F)\n\n\ndef classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:\n    return {\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n        "kappa": float(cohen_kappa_score(y_true, y_pred)),\n    }\n\n\ndef attack_success_rate(\n    model: CSPLDA,\n    X_test_raw: np.ndarray,\n    y_test: np.ndarray,\n    target_class: int,\n    target_alignment: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> float:\n    non_target = y_test != target_class\n    if non_target.sum() == 0:\n        return float("nan")\n    X_attack = add_spectral_trigger(\n        X_test_raw[non_target],\n        sfreq,\n        channel_indices,\n        frequency_hz,\n        amplitude_std,\n    )\n    X_attack_aligned = apply_alignment(X_attack, target_alignment)\n    pred = model.predict(X_attack_aligned)\n    return float(np.mean(pred == target_class))\n\n\n# -----------------------------------------------------------------------------\n# Calibration split\n# -----------------------------------------------------------------------------\n\n\ndef stratified_calibration_split(\n    y: np.ndarray, fraction: float, rng: np.random.Generator\n) -> Tuple[np.ndarray, np.ndarray]:\n    """Return calibration/test indices; leaves at least one test trial per class."""\n    if not 0.0 <= fraction < 1.0:\n        raise ValueError("fraction must satisfy 0 <= fraction < 1.")\n\n    calib: List[int] = []\n    test: List[int] = []\n    for cls in np.unique(y):\n        idx = np.flatnonzero(y == cls)\n        idx = rng.permutation(idx)\n        if fraction == 0.0:\n            n_cal = 0\n        else:\n            n_cal = int(round(fraction * len(idx)))\n            n_cal = max(1, n_cal)\n            n_cal = min(n_cal, len(idx) - 1)\n        calib.extend(idx[:n_cal].tolist())\n        test.extend(idx[n_cal:].tolist())\n\n    calib_arr = np.asarray(sorted(calib), dtype=int)\n    test_arr = np.asarray(sorted(test), dtype=int)\n    if len(test_arr) == 0:\n        raise RuntimeError("Calibration split left no held-out target data.")\n    return calib_arr, test_arr\n\n\n# -----------------------------------------------------------------------------\n# Dataset loading\n# -----------------------------------------------------------------------------\n\n\ndef load_bnci_subject(\n    subject: int, fmin: float, fmax: float\n) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, float, List[str]]:\n    """Load one subject through MOABB. Import is lazy for clearer install errors."""\n    try:\n        from moabb.datasets import BNCI2014_001\n        from moabb.paradigms import MotorImagery\n    except Exception as exc:  # pragma: no cover\n        raise RuntimeError(\n            "MOABB is required for dataset download/loading. Recommended:\\n"\n            "  python -m pip install -r requirements_colab.txt\\n"\n            "Use Python 3.11 or newer."\n        ) from exc\n\n    dataset = BNCI2014_001()\n    paradigm = MotorImagery(n_classes=4, fmin=fmin, fmax=fmax)\n    epochs, labels, meta = paradigm.get_data(\n        dataset=dataset,\n        subjects=[int(subject)],\n        return_epochs=True,\n    )\n\n    # MOABB/MNE returns EEG in volts. Do not rescale: CSP is scale-invariant enough\n    # for this benchmark, while trigger amplitude is defined relative to trial SD.\n    try:\n        X = epochs.get_data(copy=True)\n    except TypeError:  # compatibility with older MNE\n        X = epochs.get_data().copy()\n    y_text = np.asarray(labels).astype(str)\n    meta = pd.DataFrame(meta).reset_index(drop=True)\n    sfreq = float(epochs.info["sfreq"])\n    ch_names = list(epochs.ch_names)\n\n    if not (len(X) == len(y_text) == len(meta)):\n        raise RuntimeError("MOABB returned inconsistent data/label/metadata lengths.")\n    if "session" not in meta.columns:\n        raise RuntimeError("MOABB metadata does not contain a \'session\' column.")\n    return X, y_text, meta, sfreq, ch_names\n\n\n# -----------------------------------------------------------------------------\n# One subject x one calibration split\n# -----------------------------------------------------------------------------\n\n\ndef evaluate_subject_seed_fraction(\n    X: np.ndarray,\n    y: np.ndarray,\n    meta: pd.DataFrame,\n    sfreq: float,\n    ch_names: Sequence[str],\n    subject: int,\n    seed: int,\n    calibration_fraction: float,\n    alignment_mode: str,\n    target_class: int,\n    cfg: Config,\n) -> List[Dict[str, object]]:\n    sessions = sorted(meta["session"].astype(str).unique().tolist())\n    if len(sessions) < 2:\n        raise RuntimeError(f"Subject {subject}: expected >=2 sessions, got {sessions}.")\n    source_session, target_session = sessions[0], sessions[1]\n\n    src_idx = np.flatnonzero(meta["session"].astype(str).to_numpy() == source_session)\n    tgt_idx = np.flatnonzero(meta["session"].astype(str).to_numpy() == target_session)\n    Xs_raw, ys = X[src_idx], y[src_idx]\n    Xt_raw, yt = X[tgt_idx], y[tgt_idx]\n\n    rng_split = np.random.default_rng(seed)\n    cal_idx, test_idx = stratified_calibration_split(\n        yt, calibration_fraction, rng_split\n    )\n    X_cal_raw, y_cal = Xt_raw[cal_idx], yt[cal_idx]\n    X_test_raw, y_test = Xt_raw[test_idx], yt[test_idx]\n\n    motor_channels = resolve_channel_indices(ch_names, cfg.trigger_channels)\n\n    # Target alignment is explicitly separated into strict inductive vs transductive.\n    Wt = choose_target_alignment(\n        alignment_mode=alignment_mode,\n        X_source_reference=Xs_raw,\n        X_target_all=Xt_raw,\n        X_target_calibration=X_cal_raw,\n    )\n    Xcal_a = apply_alignment(X_cal_raw, Wt) if len(X_cal_raw) else X_cal_raw\n    Xtest_a = apply_alignment(X_test_raw, Wt)\n\n    rows: List[Dict[str, object]] = []\n\n    def train_and_record(\n        scenario: str,\n        X_source_train_raw: np.ndarray,\n        y_source_train: np.ndarray,\n        defense_stats: Optional[Dict[str, float]] = None,\n    ) -> None:\n        Ws = fit_euclidean_alignment(X_source_train_raw)\n        Xs_a = apply_alignment(X_source_train_raw, Ws)\n        if len(Xcal_a):\n            X_train = np.concatenate([Xs_a, Xcal_a], axis=0)\n            y_train = np.concatenate([y_source_train, y_cal], axis=0)\n        else:\n            X_train, y_train = Xs_a, y_source_train\n\n        model = CSPLDA(cfg.csp_components).fit(X_train, y_train)\n        pred = model.predict(Xtest_a)\n        metrics = classification_metrics(y_test, pred)\n        asr = attack_success_rate(\n            model=model,\n            X_test_raw=X_test_raw,\n            y_test=y_test,\n            target_class=target_class,\n            target_alignment=Wt,\n            sfreq=sfreq,\n            channel_indices=motor_channels,\n            frequency_hz=cfg.trigger_frequency_hz,\n            amplitude_std=cfg.trigger_amplitude_std,\n        )\n\n        row: Dict[str, object] = {\n            "subject": int(subject),\n            "seed": int(seed),\n            "source_session": source_session,\n            "target_session": target_session,\n            "alignment_mode": alignment_mode,\n            "calibration_fraction": float(calibration_fraction),\n            "scenario": scenario,\n            "n_source_train": int(len(X_source_train_raw)),\n            "n_target_calibration": int(len(cal_idx)),\n            "n_target_test": int(len(test_idx)),\n            "target_class": int(target_class),\n            "target_class_name": cfg.target_class_name,\n            "asr": asr,\n            **metrics,\n        }\n        if defense_stats:\n            row.update({f"defense_{k}": v for k, v in defense_stats.items()})\n        rows.append(row)\n\n    # 1) Clean upper/reference baseline.\n    train_and_record("clean", Xs_raw, ys)\n\n    # 2) Poisoned source training.\n    rng_attack = np.random.default_rng(seed + 100_000 + subject)\n    Xp, yp, poison_mask = poison_training_set(\n        X=Xs_raw,\n        y=ys,\n        target_class=target_class,\n        poison_rate=cfg.poison_rate,\n        rng=rng_attack,\n        sfreq=sfreq,\n        channel_indices=motor_channels,\n        frequency_hz=cfg.trigger_frequency_hz,\n        amplitude_std=cfg.trigger_amplitude_std,\n    )\n    train_and_record("poisoned", Xp, yp)\n\n    # 3) Phase-I generic spectral sanitizer baseline.\n    spec_scores = spectral_concentration_score(\n        Xp,\n        sfreq=sfreq,\n        fmin=cfg.fmin,\n        fmax=cfg.fmax,\n        channel_indices=motor_channels,\n    )\n    Xd, yd, keep_mask, san_stats = robust_sanitize(\n        Xp,\n        yp,\n        spec_scores,\n        z_threshold=cfg.sanitizer_z_threshold,\n        max_remove_fraction=cfg.sanitizer_max_remove_fraction,\n        min_class_samples=cfg.min_class_samples_after_sanitize,\n    )\n    removed = ~keep_mask\n    san_stats = dict(san_stats)\n    san_stats["poison_recall"] = float(\n        np.sum(removed & poison_mask) / max(1, np.sum(poison_mask))\n    )\n    san_stats["removal_precision"] = float(\n        np.sum(removed & poison_mask) / max(1, np.sum(removed))\n    )\n    san_stats["clean_false_removal_rate"] = float(\n        np.sum(removed & ~poison_mask) / max(1, np.sum(~poison_mask))\n    )\n    train_and_record("spectral_baseline", Xd, yd, san_stats)\n\n    # 4) Phase-II CRCS candidate defense.\n    # Use separately aligned source/calibration data for descriptor comparison.\n    Ws_poison = fit_euclidean_alignment(Xp)\n    Xp_desc = apply_alignment(Xp, Ws_poison)\n    if len(X_cal_raw):\n        Xcal_desc = apply_alignment(X_cal_raw, Wt)\n        ycal_desc = y_cal\n    else:\n        Xcal_desc = np.empty((0, Xp.shape[1], Xp.shape[2]), dtype=Xp.dtype)\n        ycal_desc = np.empty((0,), dtype=yp.dtype)\n\n    Xc, yc, crcs_keep, crcs_stats = calibration_referenced_consistency_screen(\n        X_source=Xp_desc,\n        y_source_claimed=yp,\n        X_reference=Xcal_desc,\n        y_reference=ycal_desc,\n        sfreq=sfreq,\n        channel_indices=motor_channels,\n        subbands=cfg.crcs_subbands,\n        max_remove_fraction=cfg.crcs_max_remove_fraction,\n        min_class_samples=cfg.crcs_min_class_samples,\n        min_reference_per_class=cfg.crcs_min_reference_per_class,\n        margin_threshold=cfg.crcs_margin_threshold,\n        spectral_weight=cfg.crcs_spectral_weight,\n        spectral_z_threshold=cfg.crcs_spectral_z_threshold,\n    )\n\n    # CRCS returned aligned kept trials; train_and_record expects raw trials and\n    # fits its own source alignment. Recover raw kept trials using the same mask.\n    Xc_raw = Xp[crcs_keep]\n    yc_raw = yp[crcs_keep]\n    crcs_removed = ~crcs_keep\n    crcs_stats = dict(crcs_stats)\n    crcs_stats["poison_recall"] = float(\n        np.sum(crcs_removed & poison_mask) / max(1, np.sum(poison_mask))\n    )\n    crcs_stats["removal_precision"] = float(\n        np.sum(crcs_removed & poison_mask) / max(1, np.sum(crcs_removed))\n    )\n    crcs_stats["clean_false_removal_rate"] = float(\n        np.sum(crcs_removed & ~poison_mask) / max(1, np.sum(~poison_mask))\n    )\n    train_and_record("crcs", Xc_raw, yc_raw, crcs_stats)\n\n    return rows\n\n\n# -----------------------------------------------------------------------------\n# Full benchmark\n# -----------------------------------------------------------------------------\n\n\ndef run_benchmark(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    validate_config(cfg)\n    out_dir = Path(cfg.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    all_rows: List[Dict[str, object]] = []\n    dataset_manifest: List[Dict[str, object]] = []\n\n    for subject in cfg.subjects:\n        print(f"\\n=== Loading subject {subject} ===", flush=True)\n        X, labels_text, meta, sfreq, ch_names = load_bnci_subject(\n            subject, cfg.fmin, cfg.fmax\n        )\n        le = LabelEncoder()\n        y = le.fit_transform(labels_text)\n        if cfg.target_class_name not in le.classes_:\n            raise RuntimeError(\n                f"Target class \'{cfg.target_class_name}\' not found. Classes={list(le.classes_)}"\n            )\n        target_class = int(le.transform([cfg.target_class_name])[0])\n\n        sessions = sorted(meta["session"].astype(str).unique().tolist())\n        dataset_manifest.append(\n            {\n                "subject": subject,\n                "n_trials": len(X),\n                "n_channels": X.shape[1],\n                "n_times": X.shape[2],\n                "sfreq": sfreq,\n                "sessions": "|".join(sessions),\n                "classes": "|".join(le.classes_.tolist()),\n                "channel_names": "|".join(ch_names),\n            }\n        )\n\n        for alignment_mode in cfg.alignment_modes:\n            for fraction in cfg.calibration_fractions:\n                for seed in cfg.seeds:\n                    print(\n                        f"Subject {subject} | alignment={alignment_mode} | "\n                        f"calibration={fraction:.2f} | seed={seed}",\n                        flush=True,\n                    )\n                    rows = evaluate_subject_seed_fraction(\n                        X=X,\n                        y=y,\n                        meta=meta,\n                        sfreq=sfreq,\n                        ch_names=ch_names,\n                        subject=subject,\n                        seed=seed,\n                        calibration_fraction=fraction,\n                        alignment_mode=alignment_mode,\n                        target_class=target_class,\n                        cfg=cfg,\n                    )\n                    all_rows.extend(rows)\n\n    raw = pd.DataFrame(all_rows)\n    if raw.empty:\n        raise RuntimeError("No results were produced.")\n\n    metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "kappa", "asr"]\n    summary = (\n        raw.groupby(["alignment_mode", "calibration_fraction", "scenario"], as_index=False)[metric_cols]\n        .agg(["mean", "std", "count"])\n    )\n    summary.columns = [\n        "_".join([str(x) for x in col if str(x) != ""]).rstrip("_")\n        for col in summary.columns.to_flat_index()\n    ]\n\n    raw_path = out_dir / "trial_level_results.csv"\n    summary_path = out_dir / "summary_results.csv"\n    manifest_path = out_dir / "dataset_manifest.csv"\n    config_path = out_dir / "config.json"\n\n    raw.to_csv(raw_path, index=False)\n    summary.to_csv(summary_path, index=False)\n    pd.DataFrame(dataset_manifest).to_csv(manifest_path, index=False)\n    config_path.write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")\n\n    subject_level = (\n        raw.groupby(\n            ["subject", "alignment_mode", "calibration_fraction", "scenario"],\n            as_index=False,\n        )[metric_cols]\n        .mean()\n    )\n    subject_level.to_csv(out_dir / "subject_level_results.csv", index=False)\n    make_statistical_tests(subject_level).to_csv(\n        out_dir / "statistical_tests.csv", index=False\n    )\n\n    diag_cols = [\n        c for c in [\n            "defense_removed_fraction",\n            "defense_poison_recall",\n            "defense_removal_precision",\n            "defense_clean_false_removal_rate",\n            "defense_reference_target_calibration",\n        ] if c in raw.columns\n    ]\n    if diag_cols:\n        diagnostics = (\n            raw[raw["scenario"].isin(["spectral_baseline", "crcs"])]\n            .groupby(\n                ["alignment_mode", "calibration_fraction", "scenario"],\n                as_index=False,\n            )[diag_cols]\n            .mean()\n        )\n        diagnostics.to_csv(out_dir / "defense_diagnostics.csv", index=False)\n\n    make_plots(raw, out_dir)\n    print(f"\\nSaved results to: {out_dir.resolve()}")\n    return raw, summary\n\n\ndef _holm_adjust(p_values: Sequence[float]) -> List[float]:\n    p = np.asarray(p_values, dtype=float)\n    if len(p) == 0:\n        return []\n    order = np.argsort(p)\n    adjusted = np.empty(len(p), dtype=float)\n    running = 0.0\n    m = len(p)\n    for rank, idx in enumerate(order):\n        val = min(1.0, (m - rank) * p[idx])\n        running = max(running, val)\n        adjusted[idx] = running\n    return adjusted.tolist()\n\n\ndef make_statistical_tests(subject_level: pd.DataFrame) -> pd.DataFrame:\n    """\n    Paired Wilcoxon tests use subjects, not repeated seed-runs, as inferential units.\n    This avoids pseudoreplication. Holm correction is applied within the produced\n    family of pairwise tests.\n    """\n    tests: List[Dict[str, object]] = []\n    comparisons = [\n        ("poisoned", "crcs", "asr"),\n        ("spectral_baseline", "crcs", "asr"),\n        ("poisoned", "crcs", "accuracy"),\n        ("clean", "crcs", "accuracy"),\n    ]\n    for mode in sorted(subject_level["alignment_mode"].unique()):\n        for frac in sorted(subject_level["calibration_fraction"].unique()):\n            block = subject_level[\n                (subject_level["alignment_mode"] == mode)\n                & (subject_level["calibration_fraction"] == frac)\n            ]\n            for a, b, metric in comparisons:\n                A = block[block["scenario"] == a][["subject", metric]].rename(\n                    columns={metric: "a"}\n                )\n                B = block[block["scenario"] == b][["subject", metric]].rename(\n                    columns={metric: "b"}\n                )\n                pair = A.merge(B, on="subject", how="inner").dropna()\n                if len(pair) < 5:\n                    continue\n                diff = pair["b"].to_numpy() - pair["a"].to_numpy()\n                try:\n                    stat, p = wilcoxon(\n                        pair["a"].to_numpy(),\n                        pair["b"].to_numpy(),\n                        zero_method="wilcox",\n                        alternative="two-sided",\n                    )\n                    stat = float(stat)\n                    p = float(p)\n                except ValueError:\n                    stat, p = float("nan"), 1.0\n                tests.append(\n                    {\n                        "alignment_mode": mode,\n                        "calibration_fraction": float(frac),\n                        "scenario_a": a,\n                        "scenario_b": b,\n                        "metric": metric,\n                        "n_subjects": int(len(pair)),\n                        "mean_a": float(pair["a"].mean()),\n                        "mean_b": float(pair["b"].mean()),\n                        "mean_difference_b_minus_a": float(np.mean(diff)),\n                        "median_difference_b_minus_a": float(np.median(diff)),\n                        "wilcoxon_stat": stat,\n                        "p_raw": p,\n                    }\n                )\n    stat_columns = [\n        "alignment_mode",\n        "calibration_fraction",\n        "scenario_a",\n        "scenario_b",\n        "metric",\n        "n_subjects",\n        "mean_a",\n        "mean_b",\n        "mean_difference_b_minus_a",\n        "median_difference_b_minus_a",\n        "wilcoxon_stat",\n        "p_raw",\n        "p_holm",\n    ]\n    if not tests:\n        # Quick mode may contain only one subject, so inferential tests are\n        # intentionally unavailable. Return headers rather than a zero-byte CSV.\n        return pd.DataFrame(columns=stat_columns)\n\n    out = pd.DataFrame(tests)\n    out["p_holm"] = _holm_adjust(out["p_raw"].to_numpy())\n    return out[stat_columns]\n\n\ndef make_plots(raw: pd.DataFrame, out_dir: Path) -> None:\n    import matplotlib.pyplot as plt\n\n    grouped = (\n        raw.groupby(["alignment_mode", "calibration_fraction", "scenario"], as_index=False)\n        .agg(accuracy=("accuracy", "mean"), asr=("asr", "mean"))\n    )\n\n    for mode in sorted(grouped["alignment_mode"].unique()):\n        gm = grouped[grouped["alignment_mode"] == mode]\n\n        fig, ax = plt.subplots(figsize=(7.5, 4.8))\n        for scenario, g in gm.groupby("scenario"):\n            g = g.sort_values("calibration_fraction")\n            ax.plot(\n                g["calibration_fraction"] * 100,\n                g["accuracy"] * 100,\n                marker="o",\n                label=scenario,\n            )\n        ax.set_xlabel("Target-session labeled calibration (%)")\n        ax.set_ylabel("Held-out target-session accuracy (%)")\n        ax.set_title(f"Cross-session accuracy vs. calibration — {mode}")\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"accuracy_vs_calibration_{mode}.png", dpi=300)\n        plt.close(fig)\n\n        fig, ax = plt.subplots(figsize=(7.5, 4.8))\n        for scenario in ["poisoned", "spectral_baseline", "crcs"]:\n            g = gm[gm["scenario"] == scenario].sort_values("calibration_fraction")\n            if not g.empty:\n                ax.plot(\n                    g["calibration_fraction"] * 100,\n                    g["asr"] * 100,\n                    marker="o",\n                    label=scenario,\n                )\n        ax.set_xlabel("Target-session labeled calibration (%)")\n        ax.set_ylabel("Targeted attack success rate, ASR (%)")\n        ax.set_title(f"Backdoor vulnerability vs. calibration — {mode}")\n        ax.set_ylim(0, 100)\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"asr_vs_calibration_{mode}.png", dpi=300)\n        plt.close(fig)\n\n\n# -----------------------------------------------------------------------------\n# Synthetic smoke test (does not download BCI data)\n# -----------------------------------------------------------------------------\n\n\ndef smoke_test() -> None:\n    print("Running deterministic synthetic smoke test...")\n    set_global_seed(7)\n    rng = np.random.default_rng(7)\n    n_trials, n_ch, n_t, sfreq = 80, 8, 500, 250.0\n    X = rng.normal(size=(n_trials, n_ch, n_t))\n    y = np.repeat(np.arange(4), n_trials // 4)\n\n    # Add class-specific weak rhythms so CSP has something non-random to learn.\n    t = np.arange(n_t) / sfreq\n    for i in range(n_trials):\n        cls = int(y[i])\n        X[i, cls % n_ch] += 0.35 * np.sin(2 * np.pi * (10 + 2 * cls) * t)\n\n    W = fit_euclidean_alignment(X)\n    Xa = apply_alignment(X, W)\n    if Xa.shape != X.shape or not np.isfinite(Xa).all():\n        raise AssertionError("Alignment smoke test failed.")\n\n    rng2 = np.random.default_rng(9)\n    Xp, yp, pm = poison_training_set(\n        X, y, target_class=0, poison_rate=0.10, rng=rng2, sfreq=sfreq,\n        channel_indices=[1, 2, 3], frequency_hz=23.0, amplitude_std=0.6\n    )\n    if pm.sum() == 0 or not np.all(yp[pm] == 0):\n        raise AssertionError("Poisoning smoke test failed.")\n\n    scores = spectral_concentration_score(Xp, sfreq, 8.0, 30.0, [1, 2, 3])\n    Xd, yd, keep, _ = robust_sanitize(Xp, yp, scores, 3.5, 0.15, 5)\n    if len(Xd) != len(yd) or len(keep) != len(Xp):\n        raise AssertionError("Sanitization smoke test failed.")\n\n    cal, test = stratified_calibration_split(y, 0.25, np.random.default_rng(3))\n    if set(cal).intersection(set(test)) or len(test) == 0:\n        raise AssertionError("Calibration split smoke test failed.")\n\n    # Synthetic CRCS test with a small clean calibration reference.\n    ref_idx = np.concatenate([np.flatnonzero(y == c)[:5] for c in np.unique(y)])\n    Xc, yc, keep_c, stats_c = calibration_referenced_consistency_screen(\n        X_source=Xp,\n        y_source_claimed=yp,\n        X_reference=X[ref_idx],\n        y_reference=y[ref_idx],\n        sfreq=sfreq,\n        channel_indices=[1, 2, 3],\n        subbands=((8, 12), (12, 16), (16, 20), (20, 24), (24, 30)),\n        max_remove_fraction=0.15,\n        min_class_samples=5,\n        min_reference_per_class=3,\n        margin_threshold=0.0,\n        spectral_weight=0.25,\n        spectral_z_threshold=2.5,\n    )\n    if len(keep_c) != len(Xp) or len(Xc) != len(yc):\n        raise AssertionError("CRCS smoke test failed.")\n\n    model = CSPLDA(6).fit(Xa[:60], y[:60])\n    pred = model.predict(Xa[60:])\n    if len(pred) != 20:\n        raise AssertionError("Classifier smoke test failed.")\n\n    # Alignment-mode smoke check.\n    Wi = choose_target_alignment("inductive", X[:40], X[40:], X[40:50])\n    Wt = choose_target_alignment("transductive", X[:40], X[40:], X[40:50])\n    if Wi.shape != Wt.shape:\n        raise AssertionError("Alignment-mode smoke test failed.")\n\n    print("Smoke test PASSED.")\n\n\ndef parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--smoke-test", action="store_true", help="Run synthetic checks only.")\n    p.add_argument("--quick", action="store_true", help="Run subject 1, seed 42, calib 50/80%, both alignment modes.")\n    p.add_argument("--output-dir", default="bci_security_results")\n    p.add_argument("--subjects", nargs="*", type=int, default=None)\n    p.add_argument("--seeds", nargs="*", type=int, default=None)\n    p.add_argument("--calibration", nargs="*", type=float, default=None)\n    p.add_argument("--alignment-modes", nargs="*", choices=["inductive", "transductive"], default=None)\n    return p.parse_args(argv)\n\n\ndef main(argv: Optional[Sequence[str]] = None) -> int:\n    args = parse_args(argv)\n    if args.smoke_test:\n        smoke_test()\n        return 0\n\n    if args.quick:\n        cfg = Config(\n            subjects=(1,),\n            seeds=(42,),\n            calibration_fractions=(0.50, 0.80),\n            alignment_modes=("inductive", "transductive"),\n            output_dir=args.output_dir,\n        )\n    else:\n        cfg = Config(\n            subjects=tuple(args.subjects) if args.subjects else tuple(range(1, 10)),\n            seeds=tuple(args.seeds) if args.seeds else (11, 22, 33, 44, 55),\n            calibration_fractions=tuple(args.calibration)\n            if args.calibration\n            else (0.0, 0.10, 0.25, 0.50, 0.75, 0.80),\n            alignment_modes=tuple(args.alignment_modes)\n            if args.alignment_modes\n            else ("inductive", "transductive"),\n            output_dir=args.output_dir,\n        )\n\n    set_global_seed(cfg.seeds[0])\n    raw, summary = run_benchmark(cfg)\n    print("\\nSummary:\\n")\n    with pd.option_context("display.max_columns", None, "display.width", 180):\n        print(summary.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
path=Path('/content/bci_security_phase2_CRCS_COLAB_MASTER_v2_2.py')
path.write_text(SCRIPT,encoding='utf-8')
digest=hashlib.sha256(SCRIPT.encode('utf-8')).hexdigest()
EXPECTED='80982193fa29a3f9c8fdbb50fe7d1333f9e15c94595590b8a3042d56df7d0d36'
assert digest==EXPECTED,(digest,EXPECTED)
print('✅ Script created:',path)
print('✅ SHA256 verified:',digest)


✅ Script created: /content/bci_security_phase2_CRCS_COLAB_MASTER_v2_2.py
✅ SHA256 verified: 80982193fa29a3f9c8fdbb50fe7d1333f9e15c94595590b8a3042d56df7d0d36


In [ ]:
# STEP 4 — COMPILE + SMOKE TEST
!python -m py_compile /content/bci_security_phase2_CRCS_COLAB_MASTER_v2_2.py
!python /content/bci_security_phase2_CRCS_COLAB_MASTER_v2_2.py --smoke-test


Running deterministic synthetic smoke test...
Smoke test PASSED.


In [ ]:
# STEP 5 — QUICK HIGH-CALIBRATION REAL-DATA TEST
!python /content/bci_security_phase2_CRCS_COLAB_MASTER_v2_2.py --quick --output-dir /content/bci_phase2_quick_v2_2



=== Loading subject 1 ===
Choosing from all possible events
/usr/local/lib/python3.13/dist-packages/moabb/datasets/download.py:97: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|██████████████████████████████████████| 42.8M/42.8M [00:00<00:00, 168GB/s]
SHA256 hash of downloaded file: 054f02e70cf9c4ada1517e9b9864f45407939c1062c6793516585c6f511d0325
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS reques

In [ ]:
# STEP 6 — SAFE QUICK-RESULT INSPECTION
from pathlib import Path
import pandas as pd
from IPython.display import display
out=Path('/content/bci_phase2_quick_v2_2')
for name in ['summary_results.csv','defense_diagnostics.csv','statistical_tests.csv']:
    p=out/name
    print(f"\n### {name}")
    if not p.exists():
        print('⚠️ File not found.'); continue
    if p.stat().st_size==0:
        print('ℹ️ Empty file; no results available for this quick-mode output.'); continue
    try:
        df=pd.read_csv(p)
    except pd.errors.EmptyDataError:
        print('ℹ️ No tabular results available.'); continue
    if df.empty:
        if name=='statistical_tests.csv':
            print('ℹ️ No inferential tests in quick mode: only one subject is evaluated.')
            print('   Statistical tests will be generated in the full 9-subject run.')
        else:
            print('ℹ️ Table has headers but no rows.')
        continue
    display(df)
print('\n✅ Step 6 inspection completed.')



### summary_results.csv


,alignment_mode,calibration_fraction,scenario,accuracy_mean,accuracy_std,accuracy_count,balanced_accuracy_mean,balanced_accuracy_std,balanced_accuracy_count,macro_f1_mean,macro_f1_std,macro_f1_count,kappa_mean,kappa_std,kappa_count,asr_mean,asr_std,asr_count
0,inductive,0.5,clean,0.812500,NaN,1,0.812500,NaN,1,0.810300,NaN,1,0.750000,NaN,1,0.000000,NaN,1
1,inductive,0.5,crcs,0.791667,NaN,1,0.791667,NaN,1,0.790343,NaN,1,0.722222,NaN,1,0.000000,NaN,1
2,inductive,0.5,poisoned,0.743056,NaN,1,0.743056,NaN,1,0.740137,NaN,1,0.657407,NaN,1,0.712963,NaN,1
3,inductive,0.5,spectral_baseline,0.750000,NaN,1,0.750000,NaN,1,0.747523,NaN,1,0.666667,NaN,1,0.712963,NaN,1
4,inductive,0.8,clean,0.803571,NaN,1,0.803571,NaN,1,0.797560,NaN,1,0.738095,NaN,1,0.000000,NaN,1
5,inductive,0.8,crcs,0.785714,NaN,1,0.785714,NaN,1,0.782583,NaN,1,0.714286,NaN,1,0.000000,NaN,1
6,inductive,0.8,poisoned,0.750000,NaN,1,0.750000,NaN,1,0.747163,NaN,1,0.666667,NaN,1,0.452381,NaN,1
7,inductive,0.8,spectral_baseline,0.750000,NaN,1,0.750000,NaN,1,0.747163,NaN,1,0.666667,NaN,1,0.452381,NaN,1
8,transductive,0.5,clean,0.819444,NaN,1,0.819444,NaN,1,0.818154,NaN,1,0.759259,NaN,1,0.000000,NaN,1
9,transductive,0.5,crcs,0.784722,NaN,1,0.784722,NaN,1,0.783594,NaN,1,0.712963,NaN,1,0.000000,NaN,1



### defense_diagnostics.csv


,alignment_mode,calibration_fraction,scenario,defense_removed_fraction,defense_poison_recall,defense_removal_precision,defense_clean_false_removal_rate,defense_reference_target_calibration
0,inductive,0.5,crcs,0.149306,1.0,0.674419,0.054054,1.0
1,inductive,0.5,spectral_baseline,0.017361,0.0,0.000000,0.019305,NaN
2,inductive,0.8,crcs,0.149306,1.0,0.674419,0.054054,1.0
3,inductive,0.8,spectral_baseline,0.017361,0.0,0.000000,0.019305,NaN
4,transductive,0.5,crcs,0.149306,1.0,0.674419,0.054054,1.0
5,transductive,0.5,spectral_baseline,0.017361,0.0,0.000000,0.019305,NaN
6,transductive,0.8,crcs,0.149306,1.0,0.674419,0.054054,1.0
7,transductive,0.8,spectral_baseline,0.017361,0.0,0.000000,0.019305,NaN



### statistical_tests.csv
ℹ️ No inferential tests in quick mode: only one subject is evaluated.
   Statistical tests will be generated in the full 9-subject run.

✅ Step 6 inspection completed.


In [ ]:
# STEP 7 — FULL PHASE-II BENCHMARK
!python /content/bci_security_phase2_CRCS_COLAB_MASTER_v2_2.py --output-dir /content/bci_phase2_full_v2_2



=== Loading subject 1 ===
Choosing from all possible events
Subject 1 | alignment=inductive | calibration=0.00 | seed=11
Subject 1 | alignment=inductive | calibration=0.00 | seed=22
Subject 1 | alignment=inductive | calibration=0.00 | seed=33
Subject 1 | alignment=inductive | calibration=0.00 | seed=44
Subject 1 | alignment=inductive | calibration=0.00 | seed=55
Subject 1 | alignment=inductive | calibration=0.10 | seed=11
Subject 1 | alignment=inductive | calibration=0.10 | seed=22
Subject 1 | alignment=inductive | calibration=0.10 | seed=33
Subject 1 | alignment=inductive | calibration=0.10 | seed=44
Subject 1 | alignment=inductive | calibration=0.10 | seed=55
Subject 1 | alignment=inductive | calibration=0.25 | seed=11
Subject 1 | alignment=inductive | calibration=0.25 | seed=22
Subject 1 | alignment=inductive | calibration=0.25 | seed=33
Subject 1 | alignment=inductive | calibration=0.25 | seed=44
Subject 1 | alignment=inductive | calibration=0.25 | seed=55
Subject 1 | alignment=in

In [ ]:
# STEP 8 — HIGH-CALIBRATION DECISION TABLE
import pandas as pd
raw=pd.read_csv('/content/bci_phase2_full_v2_2/trial_level_results.csv')
rows=[]
for mode in sorted(raw['alignment_mode'].unique()):
    for frac in [0.50,0.75,0.80]:
        b=raw[(raw['alignment_mode']==mode)&(raw['calibration_fraction']==frac)]
        m=b.groupby('scenario')[['accuracy','asr']].mean()
        clean_acc=float(m.loc['clean','accuracy']); poisoned_acc=float(m.loc['poisoned','accuracy']); poisoned_asr=float(m.loc['poisoned','asr']); crcs_acc=float(m.loc['crcs','accuracy']); crcs_asr=float(m.loc['crcs','asr'])
        rows.append({'alignment_mode':mode,'calibration_pct':int(frac*100),'clean_accuracy_pct':100*clean_acc,'poisoned_accuracy_pct':100*poisoned_acc,'poisoned_asr_pct':100*poisoned_asr,'crcs_accuracy_pct':100*crcs_acc,'crcs_asr_pct':100*crcs_asr,'crcs_clean_accuracy_drop_pp':100*(clean_acc-crcs_acc),'ASR_below_15pct':crcs_asr<0.15,'accuracy_drop_le_2pp':(clean_acc-crcs_acc)<=0.02})
from IPython.display import display
display(pd.DataFrame(rows))


,alignment_mode,calibration_pct,clean_accuracy_pct,poisoned_accuracy_pct,poisoned_asr_pct,crcs_accuracy_pct,crcs_asr_pct,crcs_clean_accuracy_drop_pp,ASR_below_15pct,accuracy_drop_le_2pp
0,inductive,50,69.953704,67.839506,81.028807,70.231481,14.958848,-0.277778,True,True
1,inductive,75,72.037037,69.783951,72.304527,71.172840,11.563786,0.864198,True,True
2,inductive,80,73.293651,69.801587,70.052910,73.095238,10.529101,0.198413,True,True
3,transductive,50,70.231481,67.716049,81.707819,70.200617,15.041152,0.030864,False,True
4,transductive,75,72.222222,69.907407,72.551440,71.296296,10.946502,0.925926,True,True
5,transductive,80,73.095238,69.920635,70.423280,72.936508,10.740741,0.158730,True,True


In [ ]:
# STEP 9 — VERIFY FULL EXPERIMENT COMPLETENESS
import pandas as pd
from pathlib import Path
out=Path('/content/bci_phase2_full_v2_2')
raw=pd.read_csv(out/'trial_level_results.csv'); stats=pd.read_csv(out/'statistical_tests.csv')
expected=9*5*6*2*4
print('Rows obtained :',len(raw)); print('Rows expected :',expected); print('Statistical-test rows:',len(stats))
assert len(raw)==expected
assert sorted(raw['calibration_fraction'].unique().round(2).tolist())==[0.0,0.1,0.25,0.5,0.75,0.8]
assert sorted(raw['subject'].unique().tolist())==list(range(1,10))
assert len(stats)>0
print('✅ Full 2160-row matrix and inferential statistics are complete.')


Rows obtained : 2160
Rows expected : 2160
Statistical-test rows: 48
✅ Full 2160-row matrix and inferential statistics are complete.


In [ ]:
# STEP 10 — ZIP AND DOWNLOAD FULL RESULTS
from pathlib import Path
import shutil
from google.colab import files
result_dir=Path('/content/bci_phase2_full_v2_2')
if not result_dir.exists(): raise FileNotFoundError('Run STEP 7 first.')
archive=shutil.make_archive('/content/BCI_SECURITY_PHASE2_CRCS_v2_2_FULL_RESULTS','zip',root_dir=result_dir)
print('✅ Created:',archive)
files.download(archive)


✅ Created: /content/BCI_SECURITY_PHASE2_CRCS_v2_2_FULL_RESULTS.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>